# CAMELYON16 feature preprocessing on Kaggle
Attach the small preprocessing bundle dataset and one raw feature dataset. Set `SOURCE` below, then run all cells. Prepare each source in a separate notebook run.

After an interactive test succeeds, choose **Save Version → Save & Run All**, wait for the saved run to finish, and inspect its Output tab. Quick Save does not persist the prepared files.


In [ ]:
SOURCE = "uniscale"  # uniscale, multiscale, or uni
SEED = 42


In [ ]:
from pathlib import Path
import json, subprocess, sys, zipfile

slugs = {
    "uniscale": "camelyon16-precomputedfeatures-uniscale",
    "multiscale": "camelyon16-precomputedfeatures-multiscale",
    "uni": "camelyon16-uni-uniscale-preprocessed-features",
}
assert SOURCE in slugs, SOURCE
input_mounts = [p for p in Path("/kaggle/input").iterdir() if p.is_dir()]
if SOURCE == "uni":
    feature_roots = [p.parent for p in Path("/kaggle/input").rglob("embeddings")
                     if p.is_dir() and (p.parent / "patches" / "patches").is_dir()]
else:
    feature_roots = [p.parent for p in Path("/kaggle/input").rglob("0-normal")
                     if p.is_dir() and (p.parent / "1-tumor").is_dir()]
    if SOURCE == "multiscale":
        feature_roots = [p for p in feature_roots if "cat" in p.parts]
    else:
        feature_roots = [p for p in feature_roots if "cat" not in p.parts and "fuse" not in p.parts]
assert len(feature_roots) == 1, f"Expected one {SOURCE} feature folder; found {feature_roots}; mounted folders: {[p.name for p in input_mounts]}"
input_root = feature_roots[0]
code_matches = [p for p in Path("/kaggle/input").rglob("prepare.py") if (p.parent / "reference_slides.csv").is_file()]
if len(code_matches) == 1:
    code_dir = code_matches[0].parent
else:
    bundles = list(Path("/kaggle/input").rglob("preprocessing_bundle.zip"))
    assert len(bundles) == 1, f"Cannot locate preprocessing code; found scripts={code_matches}, bundles={bundles}"
    code_dir = Path("/kaggle/working/camelyon16_preprocessing_code")
    code_dir.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(bundles[0]) as bundle:
        bundle.extractall(code_dir)
output_root = Path("/kaggle/working") / f"camelyon16-{SOURCE}-prepared-v1"
command = [sys.executable, str(code_dir / "prepare.py"), "--source", SOURCE, "--input-root", str(input_root), "--output-root", str(output_root), "--seed", str(SEED)]
print("Preparing", SOURCE, "from", input_root, flush=True)
subprocess.run(command, check=True)


In [ ]:
import pandas as pd
summary = json.loads((output_root / "PREPROCESSING_COMPLETE.json").read_text())
folds = pd.read_csv(output_root / "folds.csv")
slides = pd.read_csv(output_root / "slides.csv")
assert len(slides) == 399 and len(list((output_root / "bags").glob("*.npy"))) == 399
assert folds["official_split"].value_counts().to_dict() == {"train": 270, "test": 129}
assert folds.loc[folds.official_split == "test", "validation_fold"].isna().all()
print(summary)
display(pd.crosstab(folds.validation_fold, folds.label))
print("Prepared output:", output_root)
print("Choose Save Version → Save & Run All, wait for completion, then check the saved Output tab. Quick Save is insufficient.")
